In [4]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

ROOT = Path.cwd().parent
JSON_DIR = ROOT / "JSON Whole Model"

json_files = [
    "ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json",
    "ASTIDC-STAN-HE-MPD-RHP1-M-M-0001.json",
    "ASTIDC-STAN-HE-MPD-TXBP1-M-M-0001.json",
]

json_paths = [JSON_DIR / name for name in json_files]

# Show clickable file links in the notebook.
links_html = "<br>".join(
    f"<a href='file:///{path.as_posix()}' target='_blank'>{path.name}</a>"
    for path in json_paths
)
display(HTML(f"<b>Selected JSON files:</b><br>{links_html}"))

print("Category summary (duplicate counts) by JSON file:")

file_category_counts = {}
summary_rows = []

for path in json_paths:
    print("\n" + "=" * 80)
    print(f"JSON File: {path.name}")

    if not path.exists():
        missing_df = pd.DataFrame(
            [{"Category": "<missing file>", "Count": 0}]
        )
        display(missing_df)

        file_category_counts[path.name] = pd.Series(dtype="int64")
        summary_rows.append(
            {
                "JSON File": path.name,
                "Distinct Categories": 0,
                "Total Category Entries": 0,
            }
        )
        continue

    with open(path, "r", encoding="utf-8") as f:
        items = json.load(f)

    categories = []
    for item in items:
        for prop in item.get("Properties", []):
            category = str(prop.get("category", "")).strip()
            if category:
                categories.append(category)

    if categories:
        counts = pd.Series(categories, name="Category").value_counts(dropna=False)
        file_category_counts[path.name] = counts

        file_summary_df = (
            counts
            .rename_axis("Category")
            .reset_index(name="Count")
            .sort_values(["Category"], ascending=[True], kind="stable")
            .reset_index(drop=True)
        )

        summary_rows.append(
            {
                "JSON File": path.name,
                "Distinct Categories": int(file_summary_df["Category"].nunique()),
                "Total Category Entries": int(file_summary_df["Count"].sum()),
            }
        )
    else:
        file_summary_df = pd.DataFrame(
            [{"Category": "<no categories found>", "Count": 0}]
        )
        file_category_counts[path.name] = pd.Series(dtype="int64")

        summary_rows.append(
            {
                "JSON File": path.name,
                "Distinct Categories": 0,
                "Total Category Entries": 0,
            }
        )

    display(file_summary_df)

print("\n" + "=" * 80)
print("JSON-level summary table:")
json_summary_df = pd.DataFrame(summary_rows)
display(json_summary_df)

print("\n" + "=" * 80)
print("All categories across JSON files ('-' means category not present in that file):")

all_categories = sorted({
    category
    for counts in file_category_counts.values()
    for category in counts.index.tolist()
})

category_matrix_rows = []
for category in all_categories:
    row = {"Category": category}
    for file_name in json_files:
        counts = file_category_counts.get(file_name, pd.Series(dtype="int64"))
        value = counts.get(category)
        row[file_name] = int(value) if pd.notna(value) else "-"
    category_matrix_rows.append(row)

category_matrix_df = pd.DataFrame(category_matrix_rows)
display(category_matrix_df)

Category summary (duplicate counts) by JSON file:

JSON File: ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json


,Category,Count
0,ABB4HVDCDesign,168
1,ABB4HVDCDesignRevision,300
2,ABB4ModelRevision Master,52
3,DB Part,52
4,DXF_DWG,4
5,IFC,192
6,IFCAPPLICATION,3
7,IFCORGANIZATION,1
8,IFCOWNERHISTORY,3
9,IFCPERSON,3



JSON File: ASTIDC-STAN-HE-MPD-RHP1-M-M-0001.json


,Category,Count
0,ABB4HVDCDesign,40252
1,ABB4HVDCDesignRevision,53818
2,ABB4ModelRevision Master,3504
3,DB Component Instance,422
4,DB Part,3272
5,IFC,44998
6,IFCAPPLICATION,3
7,IFCORGANIZATION,1
8,IFCOWNERHISTORY,3
9,IFCPERSON,3



JSON File: ASTIDC-STAN-HE-MPD-TXBP1-M-M-0001.json


,Category,Count
0,ABB4HVDCDesign,8404
1,ABB4HVDCDesignRevision,17456
2,ABB4ModelRevision Master,1512
3,DB Component Instance,1028
4,DB Part,3072
5,IFC,10463
6,IFCAPPLICATION,3
7,IFCORGANIZATION,1
8,IFCOWNERHISTORY,3
9,IFCPERSON,3



JSON-level summary table:


,JSON File,Distinct Categories,Total Category Entries
0,ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json,16,3594
1,ASTIDC-STAN-HE-MPD-RHP1-M-M-0001.json,17,567887
2,ASTIDC-STAN-HE-MPD-TXBP1-M-M-0001.json,18,165010



All categories across JSON files ('-' means category not present in that file):


,Category,ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json,ASTIDC-STAN-HE-MPD-RHP1-M-M-0001.json,ASTIDC-STAN-HE-MPD-TXBP1-M-M-0001.json
0,ABB4HVDCDesign,168,40252,8404
1,ABB4HVDCDesignRevision,300,53818,17456
2,ABB4ModelRevision Master,52,3504,1512
3,DB Component Instance,-,422,1028
4,DB Part,52,3272,3072
5,DXF_DWG,4,-,-
6,IFC,192,44998,10463
7,IFCAPPLICATION,3,3,3
8,IFCORGANIZATION,1,1,1
9,IFCOWNERHISTORY,3,3,3


In [6]:
print("\n" + "=" * 80)
print("EXPLORING IFC DATA STRUCTURE")
print("=" * 80)

# Reload files to explore IFC properties
ifc_analysis = {}

for path in json_paths:    
    if not path.exists():
        print(f"<file not found: {path.name}>")
        continue
    
    with open(path, "r", encoding="utf-8") as f:
        items = json.load(f)
    
    # Extract all IFC properties
    ifc_properties = []
    for item in items:
        for prop in item.get("Properties", []):
            if prop.get("category") == "IFC":
                ifc_properties.append({
                    "displayName": prop.get("displayName", ""),
                    "value": prop.get("value", ""),
                })
    
    if ifc_properties:
        ifc_df = pd.DataFrame(ifc_properties)
        
        # Count occurrences of each displayName
        field_counts = ifc_df["displayName"].value_counts().sort_index()
        
        # Store for cross-file analysis
        ifc_analysis[path.name] = {
            "total_ifc_properties": len(ifc_properties),
            "unique_fields": set(field_counts.index.tolist()),
            "field_counts": field_counts
        }
    else:
        print(f"<no IFC properties found: {path.name}>")

# Unified IFC table across all JSON files
print("\nIFC Fields Summary across all JSON files:")
print("=" * 80)

all_ifc_fields = sorted({
    field
    for analysis in ifc_analysis.values()
    for field in analysis["unique_fields"]
})

ifc_matrix_rows = []
for field in all_ifc_fields:
    row = {"IFC Field": field}
    for file_name in json_files:
        counts = ifc_analysis.get(file_name, {}).get("field_counts", pd.Series(dtype="int64"))
        value = counts.get(field)
        row[file_name] = int(value) if pd.notna(value) else "-"
    ifc_matrix_rows.append(row)

ifc_matrix_df = pd.DataFrame(ifc_matrix_rows)
display(ifc_matrix_df)

# Overall IFC summary
print("\n" + "=" * 80)
print("IFC Summary Statistics:")
print("=" * 80)

ifc_summary_rows = []
for file_name in json_files:
    if file_name in ifc_analysis:
        analysis = ifc_analysis[file_name]
        ifc_summary_rows.append({
            "JSON File": file_name,
            "Total IFC Properties": analysis["total_ifc_properties"],
            "Unique IFC Fields": len(analysis["unique_fields"]),
        })
    else:
        ifc_summary_rows.append({
            "JSON File": file_name,
            "Total IFC Properties": 0,
            "Unique IFC Fields": 0,
        })

ifc_summary_df = pd.DataFrame(ifc_summary_rows)
display(ifc_summary_df)

print(f"\nTotal unique IFC fields across all files: {len(all_ifc_fields)}")


EXPLORING IFC DATA STRUCTURE

IFC Fields Summary across all JSON files:


,IFC Field,ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json,ASTIDC-STAN-HE-MPD-RHP1-M-M-0001.json,ASTIDC-STAN-HE-MPD-TXBP1-M-M-0001.json
0,ASSEMBLYPLACE,2,600,151
1,COMPOSITIONTYPE,29,6799,1567
2,DESCRIPTION,31,7399,1718
3,ELEVATIONOFREFHEIGHT,1,1,1
4,ELEVATIONOFTERRAIN,1,1,1
5,GLOBALID,31,7399,1718
6,LONGNAME,1,1,1
7,NAME,32,7400,1719
8,OBJECTTYPE,31,7399,1718
9,PHASE,1,1,1



IFC Summary Statistics:


,JSON File,Total IFC Properties,Unique IFC Fields
0,ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json,192,12
1,ASTIDC-STAN-HE-MPD-RHP1-M-M-0001.json,44998,12
2,ASTIDC-STAN-HE-MPD-TXBP1-M-M-0001.json,10463,12



Total unique IFC fields across all files: 12


In [8]:
print("\n" + "=" * 80)
print("EXPLORING ITEM AND MATERIAL FIELD STRUCTURE")
print("=" * 80)


def build_field_matrix(category_names, section_title, field_label):
    analysis = {}

    for path in json_paths:
        if not path.exists():
            print(f"<file not found: {path.name}>")
            continue

        with open(path, "r", encoding="utf-8") as f:
            items = json.load(f)

        field_rows = []
        for item in items:
            for prop in item.get("Properties", []):
                category = str(prop.get("category", "")).strip()
                if category in category_names:
                    field_rows.append(
                        {
                            "displayName": str(prop.get("displayName", "")).strip(),
                            "value": prop.get("value", ""),
                        }
                    )

        if field_rows:
            df = pd.DataFrame(field_rows)
            counts = (
                df["displayName"]
                .fillna("")
                .replace("", "<blank displayName>")
                .value_counts()
                .sort_index()
            )
            analysis[path.name] = {
                "total_properties": len(field_rows),
                "unique_fields": set(counts.index.tolist()),
                "field_counts": counts,
            }
        else:
            analysis[path.name] = {
                "total_properties": 0,
                "unique_fields": set(),
                "field_counts": pd.Series(dtype="int64"),
            }

    print(f"\n{section_title}")
    print("=" * 80)

    all_fields = sorted(
        {
            field
            for file_analysis in analysis.values()
            for field in file_analysis["unique_fields"]
        }
    )

    matrix_rows = []
    for field in all_fields:
        row = {field_label: field}
        for file_name in json_files:
            counts = analysis[file_name]["field_counts"]
            value = counts.get(field)
            row[file_name] = int(value) if pd.notna(value) else "-"
        matrix_rows.append(row)

    if matrix_rows:
        matrix_df = pd.DataFrame(matrix_rows)
    else:
        matrix_df = pd.DataFrame(columns=[field_label] + json_files)
    display(matrix_df)

    summary_rows = []
    for file_name in json_files:
        file_analysis = analysis[file_name]
        summary_rows.append(
            {
                "JSON File": file_name,
                f"Total {field_label} Properties": file_analysis["total_properties"],
                f"Unique {field_label}s": len(file_analysis["unique_fields"]),
            }
        )

    summary_df = pd.DataFrame(summary_rows)
    display(summary_df)


build_field_matrix(
    category_names={"Item"},
    section_title="Item Fields Summary across all JSON files:",
    field_label="Item Field",
)

build_field_matrix(
    category_names={"Material", "Materials"},
    section_title="Material Fields Summary across all JSON files:",
    field_label="Material Field",
)



EXPLORING ITEM AND MATERIAL FIELD STRUCTURE

Item Fields Summary across all JSON files:


,Item Field,ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json,ASTIDC-STAN-HE-MPD-RHP1-M-M-0001.json,ASTIDC-STAN-HE-MPD-TXBP1-M-M-0001.json
0,GUID,19,4002,937
1,Hidden,180,16047,6580
2,Icon,180,16047,6580
3,Material,180,16047,6580
4,Name,180,16023,6572
5,Required,180,16047,6580
6,Source File,179,16046,6579
7,Type,180,16047,6580
8,Unit,1,1,1


,JSON File,Total Item Field Properties,Unique Item Fields
0,ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json,1279,9
1,ASTIDC-STAN-HE-MPD-RHP1-M-M-0001.json,116307,9
2,ASTIDC-STAN-HE-MPD-TXBP1-M-M-0001.json,46989,9



Material Fields Summary across all JSON files:


,Material Field,ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json,ASTIDC-STAN-HE-MPD-RHP1-M-M-0001.json,ASTIDC-STAN-HE-MPD-TXBP1-M-M-0001.json
0,GLOBALID,28,6798,1566
1,NX_Area,26,6640,740
2,NX_AreaSource,26,6194,672
3,NX_Density,24,1644,730
4,NX_DensitySource,24,1198,662
5,NX_Mass,24,1686,730
6,NX_MassSource,24,1216,662
7,NX_Material,-,1432,1272
8,NX_MaterialMissingAssignments,28,6798,1566
9,NX_MaterialMultipleAssigned,28,6798,1566


,JSON File,Total Material Field Properties,Unique Material Fields
0,ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json,332,13
1,ASTIDC-STAN-HE-MPD-RHP1-M-M-0001.json,56140,14
2,ASTIDC-STAN-HE-MPD-TXBP1-M-M-0001.json,12970,14
